# Notebook 2: Movie Chatbot with Web Search

## Learning Objectives
- Add web search capability to chatbot
- Implement tool-based agentic behavior
- Understand when LLM decides to use tools
- Maintain conversation memory with tools

## Features:
- ✅ Conversational AI with memory
- ✅ Web search via SERPER API
- ✅ Agentic tool selection
- ✅ Thread-based conversation tracking

In [ ]:
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai google-search-results --quiet

In [46]:
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from typing import Literal

from ai_course_utils import load_llm_from_env, load_use_case_config, display_config

# Load environment
load_dotenv()
load_dotenv('../.env')

print("✓ Imports successful")

✓ Imports successful


In [47]:
display_config()

API CONFIGURATION (.env file)
LLM Provider:    openai
LLM Model:       gpt-4.1-mini
Temperature:     0.0

API Keys Status:
  OpenAI               ✓ Set
  Google               ✗ Not set
  Mistral              ✗ Not set
  Anthropic            ✗ Not set
  Serper (Web Search)  ✓ Set

Data Files:
  Provide file paths as function parameters
  Example: load_use_case_config('your_file.xlsx')


In [48]:
# Load use case
use_case_file = "../data/use_case_Movie_Recommendation.xlsx"
use_case_config = load_use_case_config(use_case_file)
system_prompt = use_case_config.get("agent_prompt", "You are a helpful movie assistant")

print(f"\n✓ System prompt loaded")

✓ Use case loaded: ../data/use_case_Movie_Recommendation.xlsx
  Components: user, agent_prompt, url

✓ System prompt loaded


## Define Web Search Tool

In [49]:
@tool
def search_web(query: str) -> str:
    """Search the web for current movie information, reviews, and news."""
    try:
        search = GoogleSerperAPIWrapper()
        return search.run(query)
    except Exception as e:
        return f"Search error: {str(e)}"

tools = [search_web]
print("✓ Tools configured")

✓ Tools configured


In [50]:
# Initialize LLM with tools
llm = load_llm_from_env()
llm_with_tools = llm.bind_tools(tools)

print("✓ LLM initialized")

🤖 Loading LLM: openai / gpt-4.1-mini
✓ LLM initialized


## Build Agent Graph with Memory

In [51]:
def assistant(state: MessagesState):
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(messages)]}

def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    return "tools" if state["messages"][-1].tool_calls else "__end__"

# Build graph
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("assistant", assistant)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "assistant")
graph_builder.add_conditional_edges("assistant", should_continue, ["tools", "__end__"])
graph_builder.add_edge("tools", "assistant")

# Compile WITH MEMORY
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("✓ Agent graph with memory built")

✓ Agent graph with memory built


## Chat Helper Function

In [52]:
def chat(user_input: str, thread_id: str = "default") -> str:
    """
    Chat with the movie assistant.
    
    Args:
        user_input: Your message
        thread_id: Conversation ID (use same ID to continue conversation)
    
    Returns:
        Bot's response
    """
    config = {"configurable": {"thread_id": thread_id}}
    
    result = None
    for event in graph.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config,
        stream_mode="values"
    ):
        result = event
    
    return result["messages"][-1].content

print("🎬 Movie Chatbot with Web Search Ready!")
print("\nUsage: chat('your question here')")
print("Use same thread_id to continue conversation")

🎬 Movie Chatbot with Web Search Ready!

Usage: chat('your question here')
Use same thread_id to continue conversation


## Test Conversations

In [53]:
# Conversation 1
print("="*70)
print("Conversation 1: Testing Memory")
print("="*70)

response1 = chat("What are the latest Marvel movies?", thread_id="conv1")
print(f"User: What are the latest Marvel movies?")
print(f"Bot: {response1}")

print("\n" + "-"*70 + "\n")

response2 = chat("Which one has the best reviews?", thread_id="conv1")
print(f"User: Which one has the best reviews?")
print(f"Bot: {response2}")

Conversation 1: Testing Memory
User: What are the latest Marvel movies?
Bot: The latest Marvel movies released or coming out in 2024 include:

- Echo (January 9, 2024)
- Madame Web (February 14, 2024)
- Deadpool & Wolverine (July 26, 2024)

There are also upcoming Marvel movies planned for later years, such as "Avengers: Doomsday" in 2026 and others in the Marvel Phase Five lineup. Would you like information on any specific movie or Marvel phase?

----------------------------------------------------------------------

User: Which one has the best reviews?
Bot: Among the latest Marvel movies in 2024:

- "Echo" received generally positive reviews, praised for its choreography, cinematography, and a strong performance by Vincent D'Onofrio, with a rating around 7/10.
- "Madame Web" had mixed to negative reviews, criticized for poor writing, acting, and direction, though some found it entertaining in a campy way.
- "Deadpool & Wolverine" has the best reviews overall, with an 81% fresh ratin

In [45]:
response3 = chat("What is the Genre classification difference between TMDB and IMDB for the movie Toy Story", thread_id="conv1")
print(f"User: Which one has the best reviews?")
print(f"Bot: {response3}")

User: Which one has the best reviews?
Bot: The genre classification for the movie Toy Story is as follows:

- TMDB classifies Toy Story under: Family, Comedy, Animation, and Adventure.
- IMDb classifies Toy Story under: Animation, Adventure, Comedy, Family, and Fantasy.

Both sources agree on Animation, Adventure, Comedy, and Family genres, but IMDb also includes Fantasy. Would you like to know more about Toy Story or other movies in similar genres?


In [ ]:
# New conversation (different thread)
print("\n" + "="*70)
print("Conversation 2: New Thread")
print("="*70)

response3 = chat("Recommend a family-friendly movie", thread_id="conv2")
print(f"User: Recommend a family-friendly movie")
print(f"Bot: {response3}")

## Interactive Testing

In [54]:
# YOUR TURN: Chat with the assistant
my_query = "What's playing in theaters this week?"

print(f"User: {my_query}")
print(f"Bot: {chat(my_query, thread_id='my_thread')}")

User: What's playing in theaters this week?
Bot: This week in theaters, some of the movies playing include:

- The Strangers: Chapter 3 (2026)
- Solo Mio (2026)
- Dracula (2026)
- Stray Kids: The dominATE Experience (2026)
- Whistle (2026)
- Scarlet (2026)
- Send Help
- Iron Lung
- Mercy
- Avatar: Fire and Ash
- Shelter

Are you interested in a particular genre or type of movie? Or maybe a specific actor or director? This can help me narrow down the best options for you.


## Summary

✅ **Web search capability** for current information  
✅ **Conversation memory** maintains context  
✅ **Thread-based conversations** for multiple users  
✅ **Agentic behavior** - LLM decides when to search  

**Next: Notebook 3** - Adds curated knowledge search from URLs